In [1]:
import pandas as pd
import duckdb

In [2]:
orders = pd.read_csv("data/olist_orders_dataset.csv")

In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
orders.shape

(99441, 8)

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [7]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [8]:
orders["order_id"].nunique()

99441

In [9]:
orders.duplicated().sum()

np.int64(0)

In [10]:
orders["order_purchase_timestamp"].min()

'2016-09-04 21:15:19'

In [11]:
orders["order_purchase_timestamp"].max()


'2018-10-17 17:30:18'

In [12]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

In [13]:
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [14]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [15]:
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [16]:
orders.groupby("order_status")["order_delivered_customer_date"].apply(
    lambda x: x.isna().sum()
)

order_status
approved          2
canceled        619
created           5
delivered         8
invoiced        314
processing      301
shipped        1107
unavailable     609
Name: order_delivered_customer_date, dtype: int64

In [17]:
orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


In [18]:
orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_customer_date"].notna())
].shape[0]

6

### Initial Data Quality Findings

- The orders table contains 99,441 rows and 8 columns.
- `order_id` is unique, confirming a grain of one row per order.
- No fully duplicated rows were found.
- Missing delivery timestamps are largely explained by orders that were canceled, unavailable, shipped, or still being processed.
- 8 orders are marked as delivered despite having no recorded customer delivery timestamp.
- 6 canceled orders have a recorded customer delivery timestamp, indicating possible status-history or source-system inconsistencies.

In [20]:
orders["customer_id"].nunique()

99441

In [21]:
customers = pd.read_csv("data/olist_customers_dataset.csv")

In [22]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [23]:
customers.shape

(99441, 5)

In [24]:
customers["customer_id"].nunique()

99441

In [25]:
customers["customer_unique_id"].nunique()

96096

In [26]:
customers["customer_unique_id"].value_counts().head(10)

customer_unique_id
8d50f5eadf50201ccdcedfb9e2ac8455    17
3e43e6105506432c953e165fb2acf44c     9
1b6c7548a2a1f9037c1fd3ddfed95f33     7
6469f99c1f9dfae7733b25662e7f1782     7
ca77025e7201e3b30c44b472ff346268     7
47c1a3033b8b77b3ab6e109eb4d5fdf3     6
dc813062e0fc23409cd255f7f53c7074     6
63cfc61cee11cbe306bff5857d00bfe4     6
f0e310a6839dce9de1638e0fe5ab282a     6
de34b16117594161a6a89c50b289d35a     6
Name: count, dtype: int64

In [27]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [28]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [29]:
customers.duplicated().sum()

np.int64(0)

In [30]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [31]:
customers["customer_unique_id"].duplicated().sum()

np.int64(3345)

In [32]:
customers["customer_state"].value_counts().head(10)

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

In [33]:
customers["customer_state"].nunique()

27

In [34]:
customers["customer_city"].nunique()

4119

In [35]:
customers["customer_city"].value_counts().head(10)

customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64

### Customer Table Findings

- The customers table contains 99,441 rows and 5 columns.
- `customer_id` is unique and can be used to join customers to the orders table.
- There are 96,096 distinct `customer_unique_id` values, showing that some customers placed multiple orders.
- `customer_unique_id` should therefore be used for customer-level analysis and repeat-purchase analysis.
- No fully duplicated rows or missing values were found.
- Customers are represented across 27 Brazilian federal units and 4,119 cities.
- São Paulo (SP) has the largest concentration of customer records.

In [36]:
order_items = pd.read_csv("data/olist_order_items_dataset.csv")

In [37]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [38]:
order_items.shape

(112650, 7)

In [39]:
order_items["order_id"].nunique()

98666

In [40]:
order_items["order_id"].value_counts().head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
428a2f660dc84138d969ccd69a0ab6d5    15
9ef13efd6949e4573a18964dd1bbe7f5    15
73c8ab38f07dc94389065f7eba4f297a    14
9bdc4d4c71aa1de4606060929dee888c    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
3a213fcdfe7d98be74ea0dc05a8b31ae    12
Name: count, dtype: int64

In [41]:
order_items[
    order_items["order_id"] == "8272b63d03f5f79c56e9e4120aec44ef"
]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [42]:
order_items[
    order_items["order_id"] == "8272b63d03f5f79c56e9e4120aec44ef"
]["product_id"].nunique()

3

In [43]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [44]:
order_items.duplicated().sum()

np.int64(0)

In [45]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)

In [46]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [47]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [48]:
order_items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [49]:
order_items.sort_values("price", ascending=False).head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,1,489ae2aa008f021502940f251d4cce7f,e3b4998c7a498169dc7bce44e6bb6277,2017-02-16 20:37:36,6735.00,194.31
112233,fefacc66af859508bf1a7934eab1e97f,1,69c590f7ffc7bf8db97190b6cb6ed62e,80ceebb4ee9b31afb6c6a916a574a1e2,2018-08-02 04:05:13,6729.00,193.21
107841,f5136e38d1a14a4dbd87dff67da82701,1,1bdf5e6731585cf01aa8169c7028d6ad,ee27a8f15b1dded4d213a468ba4eb391,2017-06-15 02:45:17,6499.00,227.66
74336,a96610ab360d42a2e5335a3998b4718a,1,a6492cc69376c469ab6f61d8f44de961,59417c56835dd8e2e72f91f809cd4092,2017-04-18 13:25:18,4799.00,151.34
11249,199af31afc78c699f0dbf71fb178d4d4,1,c3ed642d592594bb648ff4a04cee2747,59417c56835dd8e2e72f91f809cd4092,2017-05-09 15:50:15,4690.00,74.34
62086,8dbc85d1447242f3b127dda390d56e19,1,259037a6a41845e455183f89c5035f18,c72de06d72748d1a0dfb2125be43ba63,2018-06-28 12:36:36,4590.00,91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,1,a1beef8f3992dbd4cd8726796aa69c53,512d298ac2a96d1931b6bd30aa21f61d,2018-08-16 14:24:28,4399.87,113.45
45843,68101694e5c5dc7330c91e1bbc36214f,1,6cdf8fc1d741c76586d8b6b15e9eef30,ed4acab38528488b65a9a9c603ff024a,2018-04-05 08:27:27,4099.99,75.27
78310,b239ca7cd485940b31882363b52e6674,1,dd113cb02b2af9c8e5787e8f1f0722f6,821fb029fc6e495ca4f08a35d51e53a5,2018-08-02 08:15:14,4059.00,104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,1,6902c1962dd19d540807d0ab8fade5c6,fa1c13f2614d7b5c4749cbc52fecda94,2017-03-23 20:08:04,3999.90,17.01


In [50]:
(order_items["freight_value"] == 0).sum()

np.int64(383)

In [51]:
order_items[
    order_items["freight_value"] == 0
].head(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-04-27 09:31:35,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-03 19:11:40,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-09 23:10:46,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-26 13:31:30,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-11 18:15:13,69.9,0.0


In [52]:
order_items["product_id"].nunique()

32951

In [53]:
order_items["seller_id"].nunique()

3095

### Order Items Table Findings

- 112,650 item rows across 98,666 unique orders.
- One order can contain multiple item rows.
- 32,951 unique products and 3,095 unique sellers.
- `order_id + order_item_id` uniquely identifies an item row.
- No missing values or duplicate rows were found.
- Price and freight values contain legitimate-looking extremes; 383 item rows have zero freight.

In [54]:
products = pd.read_csv("data/olist_products_dataset.csv")

print("Shape:", products.shape)
print("\nUnique product IDs:", products["product_id"].nunique())
print("\nMissing values:")
print(products.isna().sum())

products.head()

Shape: (32951, 9)

Unique product IDs: 32951

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [55]:
category_translation = pd.read_csv(
    "data/product_category_name_translation.csv"
)

print("Shape:", category_translation.shape)
print("Missing values:")
print(category_translation.isna().sum())

category_translation.head()

Shape: (71, 2)
Missing values:
product_category_name            0
product_category_name_english    0
dtype: int64


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [56]:
sellers = pd.read_csv("data/olist_sellers_dataset.csv")

print("Shape:", sellers.shape)
print("Unique seller IDs:", sellers["seller_id"].nunique())
print("Missing values:")
print(sellers.isna().sum())

sellers.head()

Shape: (3095, 4)
Unique seller IDs: 3095
Missing values:
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [57]:
payments = pd.read_csv("data/olist_order_payments_dataset.csv")

print("Shape:", payments.shape)
print("Unique orders:", payments["order_id"].nunique())
print("Missing values:")
print(payments.isna().sum())

payments.head()

Shape: (103886, 5)
Unique orders: 99440
Missing values:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [58]:
print(payments["payment_type"].value_counts())

print("\nMaximum number of payment records for one order:")
print(payments["payment_sequential"].max())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Maximum number of payment records for one order:
29


In [59]:
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")

print("Shape:", reviews.shape)
print("Unique orders:", reviews["order_id"].nunique())
print("Missing values:")
print(reviews.isna().sum())

reviews.head()

Shape: (99224, 7)
Unique orders: 98673
Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [60]:
print("Shape:", reviews.shape)
print("Unique orders:", reviews["order_id"].nunique())

print("\nReview scores:")
print(reviews["review_score"].value_counts().sort_index())

print("\nMissing values:")
print(reviews.isna().sum())

Shape: (99224, 7)
Unique orders: 98673

Review scores:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [61]:
geo = pd.read_csv("data/olist_geolocation_dataset.csv")

print("Shape:", geo.shape)
print("Missing values:")
print(geo.isna().sum())

geo.head()

Shape: (1000163, 5)
Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [62]:
print("Shape:", geo.shape)
print("Unique postcode prefixes:", geo["geolocation_zip_code_prefix"].nunique())
print("Duplicate rows:", geo.duplicated().sum())
print("Missing values:")
print(geo.isna().sum())

Shape: (1000163, 5)
Unique postcode prefixes: 19015
Duplicate rows: 261831
Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64


In [63]:
conn = duckdb.connect("olist.duckdb")

In [64]:
conn.sql("SELECT 1 AS test")

┌───────┐
│ test  │
│ int32 │
├───────┤
│     1 │
└───────┘

In [65]:
conn.sql("""
CREATE OR REPLACE TABLE orders AS
SELECT *
FROM read_csv_auto('data/olist_orders_dataset.csv');
""")

In [66]:
conn.sql("""
SELECT *
FROM orders
LIMIT 5;
""")

┌──────────────────────────────────┬──────────────────────────────────┬──────────────┬──────────────────────────┬─────────────────────┬──────────────────────────────┬───────────────────────────────┬───────────────────────────────┐
│             order_id             │           customer_id            │ order_status │ order_purchase_timestamp │  order_approved_at  │ order_delivered_carrier_date │ order_delivered_customer_date │ order_estimated_delivery_date │
│             varchar              │             varchar              │   varchar    │        timestamp         │      timestamp      │          timestamp           │           timestamp           │           timestamp           │
├──────────────────────────────────┼──────────────────────────────────┼──────────────┼──────────────────────────┼─────────────────────┼──────────────────────────────┼───────────────────────────────┼───────────────────────────────┤
│ e481f51cbdc54678b7cc49136f2d6af7 │ 9ef432eb6251297304e76186b10a928d │ deli

In [67]:
conn.sql("""
SELECT COUNT(*) AS total_orders
FROM orders;
""")

┌──────────────┐
│ total_orders │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘

In [68]:
conn.sql("""
SELECT
    order_status,
    COUNT(*) AS order_count
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
""").df()

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [69]:
conn.sql("""
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM orders
GROUP BY order_status
ORDER BY order_count DESC;
""").df()

,order_status,order_count,percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


In [70]:
conn.sql("""
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp
FROM orders
WHERE order_status = 'canceled'
LIMIT 10;
""").df()

,order_id,customer_id,order_status,order_purchase_timestamp
0,1b9ecfe83cdc259250e1a8aca174f0ad,6d6b50b66d79f80827b6d96751528d30,canceled,2018-08-04 14:29:27
1,714fb133a6730ab81fa1d3c1b2007291,e3fe72696c4713d64d3c10afe71e75ed,canceled,2018-01-26 21:34:08
2,3a129877493c8189c59c60eb71d97c29,0913cdce793684e52bbfac69d87e91fd,canceled,2018-01-25 13:34:24
3,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39
4,ed3efbd3a87bea76c2812c66a0b32219,191984a8ba4cbb2145acb4fe35b69664,canceled,2018-09-20 13:54:16
5,0966b61e30c4a07edbd7523f59b3f3e4,2fcc597b8934d99715dbfff7909dd27f,canceled,2018-05-22 18:50:55
6,9021cf1919f615a121410790dcce848f,7acf55df0298e1d2c31200fb4f6fb93b,canceled,2018-07-04 16:05:56
7,df8282afe61008dc26c6c31011474d02,aa797b187b5466bc6925aaaa4bb3bed1,canceled,2017-03-04 12:14:30
8,a39d3db795a5cf4c8b6c9dd050f0d326,ec66df2cb66dfda07c03050470e21f69,canceled,2017-03-13 16:12:24
9,8d4c637f1accf7a88a4555f02741e606,b1dd715db389a2077f43174e7a675d07,canceled,2018-08-29 16:27:49


In [71]:
conn.sql("""
SELECT
    COUNT(*) AS canceled_orders_2018
FROM orders
WHERE order_status = 'canceled'
  AND order_purchase_timestamp >= '2018-01-01'
  AND order_purchase_timestamp < '2019-01-01';
""").df()

,canceled_orders_2018
0,334


In [72]:
conn.sql("""
SELECT
    EXTRACT(YEAR FROM order_purchase_timestamp) AS order_year,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_year
ORDER BY order_year;
""").df()

,order_year,total_orders
0,2016,329
1,2017,45101
2,2018,54011


In [73]:
conn.sql("""
SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS order_month,
    COUNT(*) AS total_orders
FROM orders
GROUP BY order_month
ORDER BY order_month;
""").df()

,order_month,total_orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [74]:
conn.sql("""
CREATE OR REPLACE TABLE order_items AS
SELECT *
FROM read_csv_auto('data/olist_order_items_dataset.csv');
""")

In [75]:
conn.sql("""
SELECT
    ROUND(SUM(price), 2) AS total_item_value
FROM order_items;
""").df()

,total_item_value
0,13591643.7


In [76]:
conn.sql("""
SELECT
    ROUND(AVG(price), 2) AS average_item_price
FROM order_items;
""").df()

,average_item_price
0,120.65


In [77]:
conn.sql("""
SELECT
    order_id,
    ROUND(SUM(price), 2) AS order_item_value,
    COUNT(*) AS item_count
FROM order_items
GROUP BY order_id
ORDER BY order_item_value DESC
LIMIT 10;
""").df()

,order_id,order_item_value,item_count
0,03caa2c082116e1d31e67e9ae3700499,13440.0,8
1,736e1922ae60d0d6a89247b851902527,7160.0,4
2,0812eb902a67711a1cb742b3cdaa65ae,6735.0,1
3,fefacc66af859508bf1a7934eab1e97f,6729.0,1
4,f5136e38d1a14a4dbd87dff67da82701,6499.0,1
5,2cc9089445046817a7539d90805e6e5a,5934.6,6
6,a96610ab360d42a2e5335a3998b4718a,4799.0,1
7,199af31afc78c699f0dbf71fb178d4d4,4690.0,1
8,b4c4b76c642808cbe472a32b86cddc95,4599.9,2
9,8dbc85d1447242f3b127dda390d56e19,4590.0,1


In [78]:
conn.sql("""
WITH order_values AS (
    SELECT
        order_id,
        SUM(price) AS order_value
    FROM order_items
    GROUP BY order_id
)

SELECT
    ROUND(AVG(order_value), 2) AS average_order_value
FROM order_values;
""").df()

,average_order_value
0,137.75


In [79]:
conn.sql("""
SELECT
    o.order_id,
    o.order_status,
    oi.order_item_id,
    oi.price
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
LIMIT 10;
""").df()

,order_id,order_status,order_item_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,delivered,1,58.90
1,00018f77f2f0320c557190d7a144bdd3,delivered,1,239.90
2,000229ec398224ef6ca0657da4fc703e,delivered,1,199.00
3,00024acbcdf0a6daa1e931b038114c75,delivered,1,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,delivered,1,199.90
5,00048cc3ae777c65dbb7d2a0634bc1ea,delivered,1,21.90
6,00054e8431b9d7675808bcb819fb4a32,delivered,1,19.90
7,000576fe39319847cbb9d288c5617fa6,delivered,1,810.00
8,0005a1a1728c9d785b8e2b08b904576c,delivered,1,145.95
9,0005f50442cb953dcd1d21e1fb923495,delivered,1,53.99


In [80]:
conn.sql("""
SELECT
    ROUND(SUM(oi.price), 2) AS delivered_item_value
FROM orders AS o
JOIN order_items AS oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered';
""").df()

,delivered_item_value
0,13221498.11


In [81]:
conn.sql("""
WITH delivered_order_values AS (
    SELECT
        o.order_id,
        SUM(oi.price) AS order_value
    FROM orders AS o
    JOIN order_items AS oi
        ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id
)

SELECT
    ROUND(AVG(order_value), 2) AS delivered_average_order_value
FROM delivered_order_values;
""").df()

,delivered_average_order_value
0,137.04


In [82]:
conn.sql("""
CREATE OR REPLACE TABLE payments AS
SELECT *
FROM read_csv_auto('data/olist_order_payments_dataset.csv');
""")

In [83]:
conn.sql("""
SELECT
    ROUND(SUM(payment_value), 2) AS total_payment_value
FROM payments;
""").df()

,total_payment_value
0,16008872.12


In [84]:
conn.sql("""
SELECT
    ROUND(SUM(price + freight_value), 2) AS total_item_plus_freight
FROM order_items;
""").df()

,total_item_plus_freight
0,15843553.24


In [85]:
conn.sql("""
WITH item_totals AS (
    SELECT
        order_id,
        SUM(price) AS item_value,
        SUM(freight_value) AS freight_value,
        SUM(price + freight_value) AS item_plus_freight
    FROM order_items
    GROUP BY order_id
)

SELECT *
FROM item_totals
LIMIT 10;
""").df()

,order_id,item_value,freight_value,item_plus_freight
0,000229ec398224ef6ca0657da4fc703e,199.00,17.87,216.87
1,0006ec9db01a64e59a68b2c340bf65a7,74.00,23.32,97.32
2,0008288aa423d2a3f00fcb17cd7d8719,99.80,26.74,126.54
3,00229e4e43f7a7e0b9dd819ad43268d3,74.90,16.49,91.39
4,00664284de7a3470d22931ed78615ee4,43.90,11.85,55.75
5,008c3a655c66f4d92370d0d422732f69,90.00,9.44,99.44
6,00c842d0ae6462afc1d38fc055b02c75,59.90,17.67,77.57
7,00d4439957b7fd3f8da4a33a965817fb,19.45,14.52,33.97
8,00eef7fe9e315a85d1a61a50cccb2494,49.90,13.37,63.27
9,00fcf938cde49ae138942b632ed62393,29.50,20.84,50.34


In [86]:
conn.sql("""
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM payments
    GROUP BY order_id
)

SELECT *
FROM payment_totals
LIMIT 10;
""").df()

,order_id,total_payment_value
0,a9810da82917af2d9aefd1278f1dcfa0,24.39
1,8ac09207f415d55acff302df7d6a895c,244.15
2,5d9c5817e278892b7498d90bfa28ade8,290.16
3,11978d520d85578c9f024b99ac1a87ef,50.09
4,c776863a93dc0740c6e7d78104b21413,264.08
5,5b7c6dbea3e21dc6314451ed73907c0c,48.27
6,c047e1042ab337c92ea23d512b54c5bf,444.25
7,0913c46b1cafecb52e5a9a55b6c0e0fc,147.00
8,940d83f3286808e4f8b1a7ac975b2aa9,76.02
9,a38d122ad3e07ca8d13e4a42059b98d2,161.62


In [87]:
conn.sql("""
WITH item_totals AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS item_plus_freight
    FROM order_items
    GROUP BY order_id
),

payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM payments
    GROUP BY order_id
)

SELECT
    i.order_id,
    ROUND(i.item_plus_freight, 2) AS item_plus_freight,
    ROUND(p.total_payment_value, 2) AS total_payment_value
FROM item_totals AS i
JOIN payment_totals AS p
    ON i.order_id = p.order_id
LIMIT 10;
""").df()

,order_id,item_plus_freight,total_payment_value
0,557cc2675910c4a6c9b54ad276f25097,56.77,56.77
1,984c8458b4e17a487d85b16c9ce0f00c,143.99,143.99
2,c596d8640962b115481b374c7b881f6a,162.83,162.83
3,d1ea13fc1406243e6f6134582bf1bbc5,1432.92,1432.92
4,cad2acef57f845758b7d0da4ee76a3e1,276.44,276.44
5,b0e1d9af1bfc3a4d96489c81cf10fe69,135.60,135.60
6,2d0da8435c6493de84783c21578638aa,76.08,76.08
7,811acfa349a56222b8ca357b548123bb,84.52,84.52
8,7ef259be178f9f06c72a57add85b71cd,372.67,372.67
9,7e54f2e2af1f04c407423998a9812c73,92.80,92.80


In [88]:
conn.sql("""
WITH item_totals AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS item_plus_freight
    FROM order_items
    GROUP BY order_id
),

payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM payments
    GROUP BY order_id
)

SELECT
    COUNT(*) AS matched_orders,
    ROUND(AVG(ABS(i.item_plus_freight - p.total_payment_value)), 2) AS avg_difference,
    ROUND(MAX(ABS(i.item_plus_freight - p.total_payment_value)), 2) AS max_difference
FROM item_totals AS i
JOIN payment_totals AS p
    ON i.order_id = p.order_id;
""").df()

,matched_orders,avg_difference,max_difference
0,98665,0.03,182.81


In [89]:
conn.sql("""
WITH item_totals AS (
    SELECT
        order_id,
        SUM(price + freight_value) AS item_plus_freight
    FROM order_items
    GROUP BY order_id
),

payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM payments
    GROUP BY order_id
)

SELECT
    i.order_id,
    ROUND(i.item_plus_freight, 2) AS item_plus_freight,
    ROUND(p.total_payment_value, 2) AS total_payment_value,
    ROUND(ABS(i.item_plus_freight - p.total_payment_value), 2) AS difference
FROM item_totals AS i
JOIN payment_totals AS p
    ON i.order_id = p.order_id
ORDER BY difference DESC
LIMIT 10;
""").df()

,order_id,item_plus_freight,total_payment_value,difference
0,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81
1,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01
2,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89
3,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53
4,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69
5,bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01
6,af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80
7,262118ce178bb3e4590a3adcf6d62e6b,177.74,126.12,51.62
8,bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43
9,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44


In [90]:
conn.sql("""
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_payment_value
    FROM payments
    GROUP BY order_id
)

SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
    ROUND(SUM(p.order_payment_value), 2) AS delivered_payment_value
FROM orders AS o
JOIN payment_totals AS p
    ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;
""").df()

,order_month,delivered_payment_value
0,2016-10-01,46566.71
1,2016-12-01,19.62
2,2017-01-01,127545.67
3,2017-02-01,271298.65
4,2017-03-01,414369.39
5,2017-04-01,390952.18
6,2017-05-01,567066.73
7,2017-06-01,490225.60
8,2017-07-01,566403.93
9,2017-08-01,646000.61


In [91]:
conn.sql("""
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_payment_value
    FROM payments
    GROUP BY order_id
),

monthly_payments AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
        SUM(p.order_payment_value) AS delivered_payment_value
    FROM orders AS o
    JOIN payment_totals AS p
        ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY order_month
)

SELECT
    order_month,
    ROUND(delivered_payment_value, 2) AS delivered_payment_value,

    ROUND(
        LAG(delivered_payment_value) OVER (ORDER BY order_month),
        2
    ) AS previous_month_value

FROM monthly_payments
ORDER BY order_month;
""").df()

,order_month,delivered_payment_value,previous_month_value
0,2016-10-01,46566.71,NaN
1,2016-12-01,19.62,46566.71
2,2017-01-01,127545.67,19.62
3,2017-02-01,271298.65,127545.67
4,2017-03-01,414369.39,271298.65
5,2017-04-01,390952.18,414369.39
6,2017-05-01,567066.73,390952.18
7,2017-06-01,490225.60,567066.73
8,2017-07-01,566403.93,490225.60
9,2017-08-01,646000.61,566403.93


In [92]:
conn.sql("""
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_payment_value
    FROM payments
    GROUP BY order_id
),

monthly_payments AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
        SUM(p.order_payment_value) AS delivered_payment_value
    FROM orders AS o
    JOIN payment_totals AS p
        ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY order_month
),

monthly_with_previous AS (
    SELECT
        order_month,
        delivered_payment_value,
        LAG(delivered_payment_value) OVER (
            ORDER BY order_month
        ) AS previous_month_value
    FROM monthly_payments
)

SELECT
    order_month,
    ROUND(delivered_payment_value, 2) AS delivered_payment_value,
    ROUND(previous_month_value, 2) AS previous_month_value,

    ROUND(
        (delivered_payment_value - previous_month_value)
        * 100.0 / previous_month_value,
        2
    ) AS month_over_month_growth_pct

FROM monthly_with_previous
ORDER BY order_month;
""").df()

,order_month,delivered_payment_value,previous_month_value,month_over_month_growth_pct
0,2016-10-01,46566.71,NaN,NaN
1,2016-12-01,19.62,46566.71,-99.96
2,2017-01-01,127545.67,19.62,649979.87
3,2017-02-01,271298.65,127545.67,112.71
4,2017-03-01,414369.39,271298.65,52.74
5,2017-04-01,390952.18,414369.39,-5.65
6,2017-05-01,567066.73,390952.18,45.05
7,2017-06-01,490225.60,567066.73,-13.55
8,2017-07-01,566403.93,490225.60,15.54
9,2017-08-01,646000.61,566403.93,14.05


In [93]:
conn.sql("""
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_payment_value
    FROM payments
    GROUP BY order_id
),

monthly_payments AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS order_month,
        SUM(p.order_payment_value) AS delivered_payment_value
    FROM orders AS o
    JOIN payment_totals AS p
        ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
    GROUP BY order_month
),

monthly_with_previous AS (
    SELECT
        order_month,
        delivered_payment_value,
        LAG(delivered_payment_value) OVER (
            ORDER BY order_month
        ) AS previous_month_value
    FROM monthly_payments
)

SELECT
    order_month,
    ROUND(delivered_payment_value, 2) AS delivered_payment_value,
    ROUND(
        (delivered_payment_value - previous_month_value)
        * 100.0 / previous_month_value,
        2
    ) AS month_over_month_growth_pct
FROM monthly_with_previous
ORDER BY order_month;
""").df()

,order_month,delivered_payment_value,month_over_month_growth_pct
0,2017-01-01,127545.67,NaN
1,2017-02-01,271298.65,112.71
2,2017-03-01,414369.39,52.74
3,2017-04-01,390952.18,-5.65
4,2017-05-01,567066.73,45.05
5,2017-06-01,490225.60,-13.55
6,2017-07-01,566403.93,15.54
7,2017-08-01,646000.61,14.05
8,2017-09-01,701169.99,8.54
9,2017-10-01,751140.27,7.13


In [94]:
conn.sql("""
CREATE OR REPLACE TABLE products AS
SELECT *
FROM read_csv_auto('data/olist_products_dataset.csv');
""")

conn.sql("""
CREATE OR REPLACE TABLE category_translation AS
SELECT *
FROM read_csv_auto('data/product_category_name_translation.csv');
""")

In [95]:
conn.sql("""
SELECT
    oi.order_id,
    oi.product_id,
    p.product_category_name,
    ct.product_category_name_english,
    oi.price
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
LIMIT 10;
""").df()

,order_id,product_id,product_category_name,product_category_name_english,price
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,cool_stuff,58.90
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,pet_shop,pet_shop,239.90
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,furniture_decor,199.00
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,perfumaria,perfumery,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,garden_tools,199.90
5,00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,housewares,21.90
6,00054e8431b9d7675808bcb819fb4a32,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,telephony,19.90
7,000576fe39319847cbb9d288c5617fa6,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,garden_tools,810.00
8,0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,health_beauty,145.95
9,0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,books_technical,53.99


In [96]:
conn.sql("""
SELECT
    ct.product_category_name_english AS category,
    COUNT(*) AS items_sold,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY total_item_value DESC
LIMIT 15;
""").df()

,category,items_sold,total_item_value,average_item_price
0,health_beauty,9670,1258681.34,130.16
1,watches_gifts,5991,1205005.68,201.14
2,bed_bath_table,11115,1036988.68,93.30
3,sports_leisure,8641,988048.97,114.34
4,computers_accessories,7827,911954.32,116.51
5,furniture_decor,8334,729762.49,87.56
6,cool_stuff,3796,635290.85,167.36
7,housewares,6964,632248.66,90.79
8,auto,4235,592720.11,139.96
9,garden_tools,4347,485256.46,111.63


In [97]:
conn.sql("""
SELECT
    ct.product_category_name_english AS category,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT oi.product_id) AS unique_products,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY total_item_value DESC
LIMIT 15;
""").df()

,category,items_sold,unique_products,total_item_value,average_item_price
0,health_beauty,9670,2444,1258681.34,130.16
1,watches_gifts,5991,1329,1205005.68,201.14
2,bed_bath_table,11115,3029,1036988.68,93.30
3,sports_leisure,8641,2867,988048.97,114.34
4,computers_accessories,7827,1639,911954.32,116.51
5,furniture_decor,8334,2657,729762.49,87.56
6,cool_stuff,3796,789,635290.85,167.36
7,housewares,6964,2335,632248.66,90.79
8,auto,4235,1900,592720.11,139.96
9,garden_tools,4347,753,485256.46,111.63


In [98]:
conn.sql("""
SELECT
    ct.product_category_name_english AS category,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT oi.product_id) AS unique_products,
    ROUND(
        COUNT(*) * 1.0 / COUNT(DISTINCT oi.product_id),
        2
    ) AS items_per_product,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY total_item_value DESC
LIMIT 15;
""").df()

,category,items_sold,unique_products,items_per_product,total_item_value,average_item_price
0,health_beauty,9670,2444,3.96,1258681.34,130.16
1,watches_gifts,5991,1329,4.51,1205005.68,201.14
2,bed_bath_table,11115,3029,3.67,1036988.68,93.30
3,sports_leisure,8641,2867,3.01,988048.97,114.34
4,computers_accessories,7827,1639,4.78,911954.32,116.51
5,furniture_decor,8334,2657,3.14,729762.49,87.56
6,cool_stuff,3796,789,4.81,635290.85,167.36
7,housewares,6964,2335,2.98,632248.66,90.79
8,auto,4235,1900,2.23,592720.11,139.96
9,garden_tools,4347,753,5.77,485256.46,111.63


In [99]:
conn.sql("""
SELECT
    oi.product_id,
    ct.product_category_name_english AS category,
    COUNT(*) AS times_sold,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
GROUP BY
    oi.product_id,
    category
ORDER BY times_sold DESC
LIMIT 15;
""").df()

,product_id,category,times_sold,total_item_value,average_item_price
0,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527,37608.90,71.36
1,99a4788cb24856965c36a24e339b6058,bed_bath_table,488,43025.56,88.17
2,422879e10f46682990de24d770e7f83d,garden_tools,484,26577.22,54.91
3,389d119b48cf3043d311335e499d9c6b,garden_tools,392,21440.59,54.70
4,368c6c730842d78016ad823897a372db,garden_tools,388,21056.80,54.27
5,53759a2ecddad2bb87a079a1f1519f73,garden_tools,373,20387.20,54.66
6,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,343,47214.51,137.65
7,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,323,37683.42,116.67
8,154e7e31ebfa092203795c972e5804a6,health_beauty,281,6325.19,22.51
9,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,274,41082.60,149.94


In [100]:
conn.sql("""
SELECT
    oi.product_id,
    ct.product_category_name_english AS category,
    COUNT(*) AS times_sold,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
GROUP BY
    oi.product_id,
    category
ORDER BY total_item_value DESC
LIMIT 15;
""").df()

,product_id,category,times_sold,total_item_value,average_item_price
0,bb50f2e236e5eea0100680137654686c,health_beauty,195,63885.00,327.62
1,6cdd53843498f92890544667809f1595,health_beauty,156,54730.20,350.83
2,d6160fb7873f184099d9bc95e30376af,computers,35,48899.34,1397.12
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,343,47214.51,137.65
4,99a4788cb24856965c36a24e339b6058,bed_bath_table,488,43025.56,88.17
5,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,274,41082.60,149.94
6,25c38557cf793876c5abdd5931f922db,baby,38,38907.32,1023.88
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,37733.90,598.95
8,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,323,37683.42,116.67
9,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527,37608.90,71.36


In [101]:
conn.sql("""
CREATE OR REPLACE TABLE sellers AS
SELECT *
FROM read_csv_auto('data/olist_sellers_dataset.csv');
""")

In [102]:
conn.sql("""
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT oi.order_id) AS orders_served,
    ROUND(SUM(oi.price), 2) AS total_item_value
FROM order_items AS oi
JOIN sellers AS s
    ON oi.seller_id = s.seller_id
GROUP BY
    oi.seller_id,
    s.seller_city,
    s.seller_state
ORDER BY total_item_value DESC
LIMIT 15;
""").df()

,seller_id,seller_city,seller_state,items_sold,orders_served,total_item_value
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1156,1132,229472.63
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,410,358,222776.05
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1987,1806,200472.92
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,586,585,194042.03
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1364,982,187923.89
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,340,336,176431.87
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1551,1314,160236.57
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1171,1160,141745.53
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1428,915,138968.55
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1499,1287,135171.70


In [103]:
conn.sql("""
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT oi.order_id) AS orders_served,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN sellers AS s
    ON oi.seller_id = s.seller_id
GROUP BY
    oi.seller_id,
    s.seller_city,
    s.seller_state
ORDER BY total_item_value DESC
LIMIT 15;
""").df()

,seller_id,seller_city,seller_state,items_sold,orders_served,total_item_value,average_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1156,1132,229472.63,198.51
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,410,358,222776.05,543.36
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1987,1806,200472.92,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,586,585,194042.03,331.13
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,1364,982,187923.89,137.77
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,340,336,176431.87,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1551,1314,160236.57,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1171,1160,141745.53,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,1428,915,138968.55,97.32
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1499,1287,135171.70,90.17


In [104]:
conn.sql("""
SELECT
    s.seller_state,
    COUNT(DISTINCT oi.seller_id) AS unique_sellers,
    COUNT(*) AS items_sold,
    COUNT(DISTINCT oi.order_id) AS orders_served,
    ROUND(SUM(oi.price), 2) AS total_item_value,
    ROUND(AVG(oi.price), 2) AS average_item_price
FROM order_items AS oi
JOIN sellers AS s
    ON oi.seller_id = s.seller_id
GROUP BY s.seller_state
ORDER BY total_item_value DESC;
""").df()

,seller_state,unique_sellers,items_sold,orders_served,total_item_value,average_item_price
0,SP,1849,80342,70188,8753396.21,108.95
1,PR,349,8671,7673,1261887.21,145.53
2,MG,244,8827,7930,1011564.74,114.60
3,RJ,171,4818,4353,843984.22,175.17
4,SC,190,4075,3667,632426.07,155.20
5,RS,129,2199,1989,378559.54,172.15
6,BA,19,643,569,285561.56,444.11
7,DF,30,899,824,97749.48,108.73
8,PE,9,448,406,91493.85,204.23
9,GO,40,520,463,66399.21,127.69


In [105]:
conn.sql("""
SELECT
    order_id,
    COUNT(DISTINCT seller_id) AS number_of_sellers
FROM order_items
GROUP BY order_id
HAVING COUNT(DISTINCT seller_id) > 1
ORDER BY number_of_sellers DESC
LIMIT 10;
""").df()

,order_id,number_of_sellers
0,cf5c8d9f52807cb2d2f0a0ff54c478da,5
1,1c11d0f4353b31ac3417fbfa5f0f2a8a,5
2,1d23106803c48c391366ff224513fb7f,4
3,8c2b13adf3f377c8f2b06b04321b0925,4
4,91be51c856a90d7efe86cf9d082d6ae3,4
5,e15b3b93174a7c20a233c102141b99fb,3
6,6d5774791f826a27e8e5521a9b816ba2,3
7,0c82a229fc22597c2361656cba8d3230,3
8,76c4c846aae2dae9e87dfa492c3f5259,3
9,62ce4e3989a3477928510bb4d1064cc4,3


In [106]:
conn.sql("""
WITH sellers_per_order AS (
    SELECT
        order_id,
        COUNT(DISTINCT seller_id) AS number_of_sellers
    FROM order_items
    GROUP BY order_id
)

SELECT
    COUNT(*) AS multi_seller_orders
FROM sellers_per_order
WHERE number_of_sellers > 1;
""").df()

,multi_seller_orders
0,1278


In [107]:
conn.sql("""
CREATE OR REPLACE TABLE customers AS
SELECT *
FROM read_csv_auto('data/olist_customers_dataset.csv');
""")

In [108]:
conn.sql("""
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS number_of_orders
    FROM customers AS c
    JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)

SELECT
    number_of_orders,
    COUNT(*) AS number_of_customers
FROM customer_orders
GROUP BY number_of_orders
ORDER BY number_of_orders;
""").df()

,number_of_orders,number_of_customers
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


In [109]:
conn.sql("""
SELECT
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS unique_customers
FROM customers
GROUP BY customer_state
ORDER BY unique_customers DESC;
""").df()

,customer_state,unique_customers
0,SP,40302
1,RJ,12384
2,MG,11259
3,RS,5277
4,PR,4882
5,SC,3534
6,BA,3277
7,DF,2075
8,ES,1964
9,GO,1952


In [110]:
conn.sql("""
WITH state_customers AS (
    SELECT
        customer_state,
        COUNT(DISTINCT customer_unique_id) AS unique_customers
    FROM customers
    GROUP BY customer_state
)

SELECT
    customer_state,
    unique_customers,
    ROUND(
        unique_customers * 100.0 /
        SUM(unique_customers) OVER (),
        2
    ) AS customer_share_pct
FROM state_customers
ORDER BY unique_customers DESC;
""").df()

,customer_state,unique_customers,customer_share_pct
0,SP,40302,41.92
1,RJ,12384,12.88
2,MG,11259,11.71
3,RS,5277,5.49
4,PR,4882,5.08
5,SC,3534,3.68
6,BA,3277,3.41
7,DF,2075,2.16
8,ES,1964,2.04
9,GO,1952,2.03


In [112]:
conn.sql("""
WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_payment_value
    FROM payments
    GROUP BY order_id
)

SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS number_of_orders,
    ROUND(SUM(p.order_payment_value), 2) AS total_customer_value,
    ROUND(AVG(p.order_payment_value), 2) AS average_order_value
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
JOIN payment_totals AS p
    ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
ORDER BY total_customer_value DESC
LIMIT 15;
""").df()

,customer_unique_id,number_of_orders,total_customer_value,average_order_value
0,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,3785.82
2,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,6929.31
4,459bef486812aa25204be022145caa62,1,6922.21,6922.21
5,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,6726.66
6,4007669dec559734d6f53e029e360987,1,6081.54,6081.54
7,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,4764.34
8,48e1ac109decbb87765a3eade6854098,1,4681.78,4681.78
9,c8460e4251689ba205045f3ea17884a1,4,4655.91,1163.98


In [113]:
conn.sql("""
SELECT
    ROUND(
        AVG(DATE_DIFF(
            'day',
            order_purchase_timestamp,
            order_delivered_customer_date
        )),
        2
    ) AS average_delivery_days,

    SUM(
        CASE
            WHEN order_delivered_customer_date > order_estimated_delivery_date
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    COUNT(*) AS delivered_orders,

    ROUND(
        SUM(
            CASE
                WHEN order_delivered_customer_date > order_estimated_delivery_date
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS late_delivery_pct

FROM orders
WHERE order_status = 'delivered'
  AND order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
""").df()

,average_delivery_days,late_orders,delivered_orders,late_delivery_pct
0,12.5,7826.0,96470,8.11


In [114]:
conn.sql("""
CREATE OR REPLACE TABLE reviews AS
SELECT *
FROM read_csv_auto('data/olist_order_reviews_dataset.csv');
""")

In [115]:
conn.sql("""
SELECT
    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Late'
        ELSE 'On Time'
    END AS delivery_status,

    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(AVG(r.review_score), 2) AS average_review_score

FROM orders AS o
JOIN reviews AS r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL

GROUP BY delivery_status;
""").df()

,delivery_status,orders,average_review_score
0,On Time,88163,4.29
1,Late,7661,2.57


In [116]:
conn.sql("""
SELECT
    review_score,
    COUNT(*) AS number_of_reviews,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS review_share_pct
FROM reviews
GROUP BY review_score
ORDER BY review_score;
""").df()

,review_score,number_of_reviews,review_share_pct
0,1,11424,11.51
1,2,3151,3.18
2,3,8179,8.24
3,4,19142,19.29
4,5,57328,57.78


In [117]:
conn.sql("""
CREATE OR REPLACE VIEW order_summary AS

WITH payment_totals AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment_value
    FROM payments
    GROUP BY order_id
),

item_totals AS (
    SELECT
        order_id,
        SUM(price) AS item_value,
        SUM(freight_value) AS freight_value,
        COUNT(*) AS items_sold
    FROM order_items
    GROUP BY order_id
),

review_totals AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score
    FROM reviews
    GROUP BY order_id
)

SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,

    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    i.items_sold,
    i.item_value,
    i.freight_value,
    p.total_payment_value,

    DATE_DIFF(
        'day',
        o.order_purchase_timestamp,
        o.order_delivered_customer_date
    ) AS delivery_days,

    CASE
        WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
        THEN 'Late'
        WHEN o.order_delivered_customer_date IS NOT NULL
        THEN 'On Time'
        ELSE NULL
    END AS delivery_status,

    r.review_score

FROM orders AS o

LEFT JOIN customers AS c
    ON o.customer_id = c.customer_id

LEFT JOIN item_totals AS i
    ON o.order_id = i.order_id

LEFT JOIN payment_totals AS p
    ON o.order_id = p.order_id

LEFT JOIN review_totals AS r
    ON o.order_id = r.order_id;
""")

In [118]:
conn.sql("""
SELECT *
FROM order_summary
LIMIT 10;
""").df()

,order_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,items_sold,item_value,freight_value,total_payment_value,delivery_days,delivery_status,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,1,29.99,8.72,38.71,8,On Time,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,1,118.70,22.76,141.46,14,On Time,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,1,159.90,19.22,179.12,9,On Time,5.0
3,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,1,19.90,8.72,28.62,3,On Time,5.0
4,a4591c265e18cb1dcee52889e2d8acc3,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-26 10:57:55,2017-08-01,1,147.90,27.36,175.26,17,On Time,4.0
5,136cce7faa42fdb2cefd53fdc79a6098,36edbb3fb164b1f16485364b6fb04c73,santa rosa,RS,invoiced,2017-04-11 12:22:08,NaT,2017-05-09,1,49.90,16.05,65.95,<NA>,NaN,2.0
6,6514b8ad8028c9f2cc2374ded245783f,932afa1e708222e5821dac9cd5db4cae,nilopolis,RJ,delivered,2017-05-16 13:10:30,2017-05-26 12:55:51,2017-06-07,1,59.99,15.17,75.16,10,On Time,5.0
7,76c6e866289321a7c93b82b54852dc33,39382392765b6dc74812866ee5ee92a7,faxinalzinho,RS,delivered,2017-01-23 18:29:09,2017-02-02 14:08:10,2017-03-06,1,19.90,16.05,35.95,10,On Time,1.0
8,e69bfb5eb88e0ed6a785585b27e16dbf,299905e3934e9e181bfb2e164dd4b4f8,sorocaba,SP,delivered,2017-07-29 11:55:02,2017-08-16 17:14:30,2017-08-23,1,149.99,19.77,169.76,18,On Time,5.0
9,e6ce16cb79ec1d90b1da9085a6118aeb,f2a85dec752b8517b5e58a06ff3cd937,rio de janeiro,RJ,delivered,2017-05-16 19:41:10,2017-05-29 11:18:31,2017-06-07,2,198.00,61.06,259.06,13,On Time,1.0


In [119]:
conn.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM order_summary;
""").df()

,total_rows,unique_orders
0,99441,99441


In [120]:
conn.sql("""
CREATE OR REPLACE VIEW sales_detail AS

SELECT
    oi.order_id,
    oi.order_item_id,

    o.order_purchase_timestamp,
    o.order_status,

    oi.product_id,
    ct.product_category_name_english AS category,

    oi.seller_id,
    s.seller_city,
    s.seller_state,

    oi.price,
    oi.freight_value

FROM order_items AS oi

JOIN orders AS o
    ON oi.order_id = o.order_id

LEFT JOIN products AS p
    ON oi.product_id = p.product_id

LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name

LEFT JOIN sellers AS s
    ON oi.seller_id = s.seller_id;
""")

In [121]:
conn.sql("""
SELECT *
FROM sales_detail
LIMIT 10;
""").df()

,order_id,order_item_id,order_purchase_timestamp,order_status,product_id,category,seller_id,seller_city,seller_state,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,2017-09-13 08:59:02,delivered,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,48436dade18ac8b2bce089ec2a041202,volta redonda,SP,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,2017-04-26 10:53:06,delivered,e5f2d52b802189ee658865ca93d83a8f,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,sao paulo,SP,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,2018-01-14 14:33:31,delivered,c777355d18b72b67abbeef9df44fd0fd,furniture_decor,5b51032eddd242adc84c38acab88f23d,borda da mata,MG,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,2018-08-08 10:00:35,delivered,7634da152a4610f1595efa32f14722fc,perfumery,9d7a1d34a5052409006425275ba1c2b4,franca,SP,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,2017-02-04 13:57:51,delivered,ac6c3623068f30de03045865e4e10089,garden_tools,df560393f3a51e74553ab94004ba5c87,loanda,PR,199.90,18.14
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,2017-05-15 21:42:34,delivered,ef92defde845ab8450f9d70c526ef70f,housewares,6426d21aca402a131fc0a5d0960a3c90,ribeirao preto,SP,21.90,12.69
6,00054e8431b9d7675808bcb819fb4a32,1,2017-12-10 11:53:48,delivered,8d4f2bb7e93e6710a28f34fa83ee7d28,telephony,7040e82f899a04d1b434b795a43b4617,sao paulo,SP,19.90,11.85
7,000576fe39319847cbb9d288c5617fa6,1,2018-07-04 12:08:27,delivered,557d850972a7d6f792fd18ae1400d9b6,garden_tools,5996cddab893a4652a15592fb58ab8db,presidente prudente,SP,810.00,70.75
8,0005a1a1728c9d785b8e2b08b904576c,1,2018-03-19 18:40:33,delivered,310ae3c140ff94b03219ad0adc3c778f,health_beauty,a416b6a846a11724393025641d4edd5e,sao paulo,SP,145.95,11.65
9,0005f50442cb953dcd1d21e1fb923495,1,2018-07-02 13:59:39,delivered,4535b0e1091c278dfd193e5a1d63b39f,books_technical,ba143b05f0110f0dc71ad71b4466ce92,sao paulo,SP,53.99,11.40


In [122]:
conn.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (order_id, order_item_id)) AS unique_order_items
FROM sales_detail;
""").df()

,total_rows,unique_order_items
0,112650,112650


In [123]:
conn.sql("""
COPY order_summary
TO 'order_summary.csv'
(HEADER, DELIMITER ',');
""")

conn.sql("""
COPY sales_detail
TO 'sales_detail.csv'
(HEADER, DELIMITER ',');
""")